In [0]:
from pyspark.sql.functions import col, current_timestamp


In [0]:
source_path = "s3://good-cabs-db/data-store/city"
target_table = "transportation.bronze.city"

In [0]:
# read raw csv data

df = (
    spark.read \
        .format("csv") \
        .option("header", True) \
        .option("inferSchema", True) \
        .option("mergeSchema", True) \
        .option("mode", "PERMISSIVE") \
        .option("columnNameOfCourptRecord", "corrupt_record")
        .load(source_path)
)

#Add metadata columns
df = (
    df \
        .withColumn("file_naem", col("_metadata.file_path"))
        .withColumn("ingest_datetime", current_timestamp())
)
# write to delta table

(
    df.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", True) \
        .saveASTable(target_table)
)

# set table properties
spark.sql(f"""
          alter table {target_table} set tblproperties (
            'qualtity' = 'bronze',
            'layer' = 'bronze',
            'source_format' = 'csv',
            'delta.enableChangeDataFeed' = 'true',
            'delta.autoOptimise.optimiseWrite' = 'true',
            'delta.autoOptimise.autoCompact' = 'true'
          """)